In [1]:
%%writefile app.py
import streamlit as st

# ==========================================
# 1. KNOWLEDGE BASE: VARIABLES & "WHY" EXPLANATIONS
# ==========================================
# This dictionary maps the variable to the UI label and explains WHY the system asks it.
QUESTIONS = {
    "Engine & Mechanical": {
        'eng_light': ("Engine Light ON", "To check for general engine faults, fuel injector issues, or sensor failures."),
        'rpm_drop': ("RPM Drop", "To diagnose fuel injector failures or timing chain jumps."),
        'trim': ("Fuel Trim > 15%", "High fuel trim indicates a potential vacuum leak or air-fuel mixture issue."),
        'maf': ("MAF Reading Low", "To identify a dirty or failing Mass Air Flow sensor."),
        'stall': ("Engine Stalling", "Critical symptom for throttle body, fuel pump, or fuel filter issues."),
        'idle': ("Idle < 500 RPM", "Low idle points towards throttle body obstructions."),
        'fuel_p': ("Fuel Pressure Low", "To confirm fuel pump failure or fuel delivery issues."),
        'misfire': ("Misfire Detected", "Used to diagnose ignition breakdowns or clogged injectors."),
        'gap': ("Spark Plug Gap Wide", "Helps confirm ignition breakdown issues."),
        'pulse': ("Injector Pulse Weak", "Points directly to a clogged or failing fuel injector."),
        'temp': ("Temp > 110C", "Critical for finding coolant leaks, dead fans, or thermostat failures."),
        'coolant': ("Coolant Level Low", "Used to diagnose coolant leaks or heater core issues."),
        'oil_p': ("Oil Pressure Low", "Identifies oil bleeding or oil pump failure."),
        'oil_lvl': ("Oil Level Normal", "Helps differentiate between a severe leak and a mechanical pump failure."),
        'knock': ("Knocking Sound", "Used to detect lubrication starvation or timing chain issues.")
    },
    "Transmission": {
        'slip': ("Gear Slip", "Primary indicator of worn clutches or burnt transmission fluid."),
        'trans_low': ("Trans Fluid Low", "Diagnoses hydraulic pressure loss or torque converter failure."),
        'burnt': ("Fluid Burnt Smell", "Confirms burnt friction plates in the transmission."),
        'harsh': ("Harsh Shift", "Points to hydraulic pressure loss or solenoid sabotage."),
        'solenoid': ("No Solenoid Response", "Identifies electronic transmission failures."),
        'trans_temp': ("Trans Temp High", "Used to diagnose overheated fluid from heavy loads."),
        'load': ("Heavy Load/Towing", "Contextual question to explain high transmission temperatures."),
        'no_drive': ("No Drive Engagement", "Checks for total torque converter failure."),
        'vibr_accel': ("Vibration on Accel", "Used to diagnose driveline imbalance or broken mounts.")
    },
    "Brakes & Wheels": {
        'click': ("Clicking on Turn", "Classic symptom used to diagnose a torn CV boot or ruined CV joint."),
        'boot': ("CV Boot Torn", "Visual confirmation needed for CV joint failure."),
        'brk_light': ("Brake Light ON", "Checks for fluid leaks, master cylinder failure, or shared sensor faults."),
        'brk_low': ("Brake Fluid Low", "Confirms a brake fluid leak."),
        'sink': ("Pedal Sinking", "Primary symptom of master cylinder internal failure."),
        'squeal': ("Brake Squeal", "Used to diagnose worn out brake pads or glazed rotors."),
        'pads': ("Pads Low", "Visual confirmation for pad wear out."),
        'hard_p': ("Hard Brake Pedal", "Diagnoses glazed rotors, ABS pump failure, or steering rack issues."),
        'spongy': ("Spongy Pedal", "Indicates air trapped in the brake lines."),
        'bubbles': ("Air Bubbles in Fluid", "Confirms air in brake lines."),
        'pulsation': ("Pedal Pulsation", "Used alongside shaking to diagnose warped rotors."),
        'shake': ("Shake at High Speed", "Diagnoses warped rotors, tie rod play, or wheel bearing issues."),
        'pull': ("Vehicle Pulls Aside", "Points to a seized brake caliper."),
        'burning': ("Burning Smell", "Confirms a seized caliper dragging on the rotor."),
        'abs_light': ("ABS Light ON", "Diagnoses ABS sensor or hydraulic pump failure."),
        'wheel_err': ("Wheel Speed Err", "Checks for ABS sensor failures or system paralysis."),
        'abs_pump': ("ABS Pump Noisy", "Confirms hydraulic ABS pump failure.")
    },
    "Electrical": {
        'bat_light': ("Battery Light ON", "Checks for alternator undercharging or diode failures."),
        'volt': ("Voltage < 13V", "Confirms the alternator is failing to charge the system."),
        'ac_leak': ("AC Leakage detected", "Diagnoses a failed alternator diode bridge."),
        'no_crank': ("No Cranking", "Used to check for a dead starter solenoid."),
        'click_start': ("Clicking when starting", "Confirms starter solenoid issues."),
        'slow_crank': ("Slow Cranking", "Diagnoses battery resistance, dead battery, or fuel delivery issues."),
        'hot_term': ("Hot Battery Terminals", "Confirms high resistance in battery cables."),
        'dead_bat': ("Dead Battery", "Checks for parasitic drain or total charging system collapse."),
        'draw': ("Current Draw High", "Confirms a parasitic electrical drain."),
        'ghost': ("Ghost Lights", "Points to a glitching Body Control Module (BCM)."),
        'lock_err': ("Lock Errors", "Confirms BCM glitches."),
        'no_dash': ("No Power to Dash", "Used to diagnose an ignition switch short."),
        'key_stuck': ("Key Stuck in Ignition", "Confirms ignition switch failures.")
    },
    "Exhaust, Fuel & Cooling": {
        'smoke_white': ("White Exhaust Smoke", "Primary indicator of a blown head gasket."),
        'smoke_black': ("Black Exhaust Smoke", "Indicates a failed fuel pressure regulator (running rich)."),
        'sulfur': ("Sulfur Egg Smell", "Diagnoses a clogged catalytic converter."),
        'loss_pwr': ("Major Loss of Power", "Used alongside sulfur smell to confirm catalytic failure."),
        'rich': ("Running Rich", "Checks for a lazy O2 sensor."),
        'mpg': ("Poor Fuel MPG", "Confirms O2 sensor failure."),
        'rough_idle': ("Rough Idle", "Diagnoses an EGR valve stuck open."),
        'stall_stop': ("Stall at Stop", "Confirms EGR valve issues."),
        'gas_smell': ("Gas Smell", "Points to an EVAP system leak."),
        'cap_light': ("Gas Cap Light", "Confirms an EVAP leak."),
        'ticking': ("Ticking Sound", "Used to diagnose an exhaust manifold leak."),
        'cabin_smell': ("Exhaust in Cabin", "Confirms an exhaust manifold leak."),
        'quiet_choke': ("Engine Choking", "Points to a collapsed muffler."),
        'back_p': ("High Backpressure", "Confirms a muffler collapse."),
        'coolant_loss': ("Coolant Loss (Internal)", "Used to diagnose head gasket breaches or internal block leaks.")
    },
    "Steering & Suspension": {
        'bouncy': ("Bouncy Ride", "Diagnoses internal strut failure."),
        'strut_leak': ("Oil on Strut", "Confirms strut failure."),
        'clunk': ("Clunk on Bumps", "Used to check control arms, ball joints, or sway bar links."),
        'wander': ("Steering Wanders", "Confirms dead control arm bushings."),
        'abs_fault': ("Electronic ABS Fault", "Used to diagnose complete traction system paralysis."),
        'traction_fault': ("Traction Control Fault", "Confirms system paralysis."),
        'whine': ("Power Steering Whine", "Diagnoses power steering pump cavitation.")
    }
}

# ==========================================
# 2. THE INFERENCE ENGINE CORE
# ==========================================
def evidence_cf(*conditions):
    return min(conditions) if conditions else 0

def apply_rule(evidence, rule_cf=0.7):
    return evidence * rule_cf

def combine_cf(cf_list):
    if not cf_list: return 0
    combined = cf_list[0]
    for next_cf in cf_list[1:]:
        combined = combined + next_cf * (1 - combined)
    return combined

# ==========================================
# 3. UI SETUP & STATE MANAGEMENT
# ==========================================
st.set_page_config(page_title="AutoSleuth KBS", layout="wide")
st.title("🕵️‍♂️ AutoSleuth: Knowledge-Based Diagnostic System")
st.markdown("This system uses Forward Chaining inference to diagnose vehicle faults based on 73 logical rules.")

# Dictionary to store user inputs
p = {var: 0.0 for cat in QUESTIONS.values() for var in cat.keys()}

st.header("Step 1: Where are you experiencing issues?")
st.write("Select the symptom categories to narrow down the questions. (This satisfies the 'Ask relevant questions' requirement).")

cols = st.columns(3)
selected_categories = []
for i, category in enumerate(QUESTIONS.keys()):
    if cols[i % 3].checkbox(category, value=(i==0)): # Default open first one
        selected_categories.append(category)

st.divider()

if selected_categories:
    st.header("Step 2: Provide specific symptoms")

    # Dynamically generate questions based ONLY on selected categories
    for cat in selected_categories:
        with st.expander(f"⚙️ {cat} Symptoms", expanded=True):
            cat_cols = st.columns(2)
            for i, (var, (label, why_text)) in enumerate(QUESTIONS[cat].items()):
                col = cat_cols[i % 2]
                p[var] = col.slider(label, 0.0, 1.0, 0.0, key=var)
                # Explanation Facility: WHY a question is asked
                with col.expander("❓ Why ask this?"):
                    st.caption(why_text)

# ==========================================
# 4. RULE EVALUATION & TRACING
# ==========================================
if st.button("RUN FORWARD CHAINING INFERENCE 🔍", type="primary", use_container_width=True):
    results = {}
    rule_text = {}
    trace = [] # Stores the step-by-step reasoning
    R = 0.7

    trace.append("▶️ **INITIALIZING FORWARD CHAINING ENGINE...**")
    trace.append("▶️ Extracting known facts from user input...")

    # Helper function to evaluate rules and log the reasoning step-by-step
    def eval_rule(rule_num, fault, cf, logic):
        trace.append(f"Evaluating Rule {rule_num}: Checking conditions for [{fault}] -> ({logic})")
        if cf > 0:
            if fault not in results: results[fault] = []
            results[fault].append(cf)
            rule_text[fault] = logic
            trace.append(f"   🟢 **TRIGGERED**: Conditions met! Added evidence for {fault} (CF: {cf:.2f})")
        else:
            trace.append(f"   🔴 **SKIPPED**: Conditions not met.")

    # --- ENGINE RULES (1-15) ---
    eval_rule(1, "Fuel Injector Suspect", apply_rule(evidence_cf(p['eng_light'], p['rpm_drop']), R), "IF Light + RPM Drop")
    eval_rule(2, "Vacuum Leak Suspect", apply_rule(evidence_cf(p['eng_light'], p['trim']), R), "IF Light + Trim > 15%")
    eval_rule(3, "Dirty MAF Sensor", apply_rule(evidence_cf(p['eng_light'], p['maf']), R), "IF Light + MAF Low")
    eval_rule(4, "Throttle Body Obstruction", apply_rule(evidence_cf(p['stall'], p['idle']), R), "IF Stall + Idle < 500")
    eval_rule(5, "Fuel Pump Dying", apply_rule(evidence_cf(p['stall'], p['fuel_p']), R), "IF Stall + Pressure Low")
    eval_rule(6, "Ignition Breakdown", apply_rule(evidence_cf(p['misfire'], p['gap']), R), "IF Misfire + Wide Gap")
    eval_rule(7, "Clogged Injector", apply_rule(evidence_cf(p['misfire'], p['pulse']), R), "IF Misfire + Weak Pulse")
    eval_rule(8, "Coolant Escape", apply_rule(evidence_cf(p['temp'], p['coolant']), R), "IF Temp > 110C + Coolant Low")
    eval_rule(9, "Cooling Fan Dead", apply_rule(evidence_cf(p['temp'], (1-p['temp'])), R), "IF Temp High + Fan RPM 0")
    eval_rule(10, "Thermostat Frozen", apply_rule(evidence_cf(p['temp'], (1-p['temp'])), R), "IF Temp High + Thermo Closed")
    eval_rule(11, "Oil Bleeding Out", apply_rule(evidence_cf(p['oil_p'], (1-p['oil_lvl'])), R), "IF Oil P Low + Oil Level Low")
    eval_rule(12, "Oil Pump Cardiac Arrest", apply_rule(evidence_cf(p['oil_p'], p['oil_lvl']), R), "IF Oil P Low + Oil Level Normal")
    eval_rule(13, "Lubrication Starvation", apply_rule(evidence_cf(p['knock'], (1-p['oil_lvl'])), R), "IF Knock + Wrong Viscosity")
    eval_rule(14, "Timing Chain Jumped", apply_rule(evidence_cf(p['knock'], p['rpm_drop']), R), "IF Knock + Timing Off")
    eval_rule(15, "Fuel Delivery DOA", apply_rule(evidence_cf(p['slow_crank'], p['fuel_p']), R), "IF Crank + No Start + No Pressure")

    # --- TRANSMISSION RULES (16-23) ---
    eval_rule(16, "Slipping Clutches", apply_rule(evidence_cf(p['slip'], p['trans_low']), R), "IF Slip + Fluid Low")
    eval_rule(17, "Burnt Friction Plates", apply_rule(evidence_cf(p['slip'], p['burnt']), R), "IF Slip + Fluid Burnt")
    eval_rule(18, "Hydraulic Pressure Loss", apply_rule(evidence_cf(p['harsh'], p['trans_low']), R), "IF Harsh + Fluid Low")
    eval_rule(19, "Solenoid Sabotage", apply_rule(evidence_cf(p['harsh'], p['solenoid']), R), "IF Harsh + No Response")
    eval_rule(20, "Overheated Trans Fluid", apply_rule(evidence_cf(p['trans_temp'], p['load']), R), "IF Trans Temp + Load")
    eval_rule(21, "Torque Converter Failure", apply_rule(evidence_cf(p['no_drive'], p['trans_low']), R), "IF No Drive + Fluid Low")
    eval_rule(22, "Driveline Imbalance", apply_rule(evidence_cf(p['vibr_accel'], p['trans_low']), R), "IF Vibr + Mount Broken")
    eval_rule(23, "CV Joint Murder", apply_rule(evidence_cf(p['click'], p['boot']), R), "IF Click + Boot Torn")

    # --- BRAKE RULES (24-32) ---
    eval_rule(24, "Brake Fluid Leak", apply_rule(evidence_cf(p['brk_light'], p['brk_low']), R), "IF Light + Fluid Low")
    eval_rule(25, "Master Cylinder Failure", apply_rule(evidence_cf(p['brk_light'], p['sink']), R), "IF Light + Pedal Sink")
    eval_rule(26, "Pad Wear Out", apply_rule(evidence_cf(p['squeal'], p['pads']), R), "IF Squeal + Pads Low")
    eval_rule(27, "Glazed Rotors", apply_rule(evidence_cf(p['squeal'], p['hard_p']), R), "IF Squeal + Hard Pedal")
    eval_rule(28, "Air in Brake Lines", apply_rule(evidence_cf(p['spongy'], p['bubbles']), R), "IF Spongy + Bubbles")
    eval_rule(29, "Warped Rotors", apply_rule(evidence_cf(p['pulsation'], p['shake']), R), "IF Pulsation + Shake")
    eval_rule(30, "Seized Caliper", apply_rule(evidence_cf(p['pull'], p['burning']), R), "IF Pull + Burning")
    eval_rule(31, "ABS Sensor Failure", apply_rule(evidence_cf(p['abs_light'], p['wheel_err']), R), "IF ABS Light + Wheel Err")
    eval_rule(32, "Hydraulic Pump Failure", apply_rule(evidence_cf(p['abs_light'], p['abs_pump']), R), "IF ABS Light + Hard Pedal")

    # --- ELECTRICAL RULES (33-39) ---
    eval_rule(33, "Alternator Undercharging", apply_rule(evidence_cf(p['bat_light'], p['volt']), R), "IF Bat Light + Volt < 13")
    eval_rule(34, "Diode Bridge Failure", apply_rule(evidence_cf(p['bat_light'], p['ac_leak']), R), "IF Bat Light + AC Leak")
    eval_rule(35, "Starter Solenoid Dead", apply_rule(evidence_cf(p['no_crank'], p['click_start']), R), "IF No Crank + Clicking")
    eval_rule(36, "Battery Cable Resistance", apply_rule(evidence_cf(p['slow_crank'], p['hot_term']), R), "IF Slow Crank + Hot Term")
    eval_rule(37, "Parasitic Drain", apply_rule(evidence_cf(p['dead_bat'], p['draw']), R), "IF Dead Bat + High Draw")
    eval_rule(38, "Body Control Module Glitch", apply_rule(evidence_cf(p['ghost'], p['lock_err']), R), "IF Ghost Light + Lock Err")
    eval_rule(39, "Ignition Switch Short", apply_rule(evidence_cf(p['no_dash'], p['key_stuck']), R), "IF No Dash + Key Stuck")

    # --- EXHAUST & EMISSION RULES (40-46) ---
    eval_rule(40, "Catalytic Converter Clogged", apply_rule(evidence_cf(p['sulfur'], p['loss_pwr']), R), "IF Sulfur + No Power")
    eval_rule(41, "O2 Sensor Lazy", apply_rule(evidence_cf(p['rich'], p['mpg']), R), "IF Rich + Poor MPG")
    eval_rule(42, "EGR Valve Stuck Open", apply_rule(evidence_cf(p['rough_idle'], p['stall_stop']), R), "IF Rough Idle + Stall")
    eval_rule(43, "EVAP System Leak", apply_rule(evidence_cf(p['gas_smell'], p['cap_light']), R), "IF Gas Smell + Cap Light")
    eval_rule(44, "Exhaust Manifold Leak", apply_rule(evidence_cf(p['ticking'], p['cabin_smell']), R), "IF Ticking + Cabin Smell")
    eval_rule(45, "Muffler Collapse", apply_rule(evidence_cf(p['quiet_choke'], p['back_p']), R), "IF Quiet Choke + Back P")
    eval_rule(46, "Head Gasket Breach", apply_rule(evidence_cf(p['smoke_white'], p['coolant_loss']), R), "IF White Smoke + Coolant Loss")

    # --- STEERING & SUSPENSION (47-54) ---
    eval_rule(47, "Strut Internal Failure", apply_rule(evidence_cf(p['bouncy'], p['strut_leak']), R), "IF Bouncy + Leak")
    eval_rule(48, "Control Arm Bushing Dead", apply_rule(evidence_cf(p['clunk'], p['wander']), R), "IF Clunk + Wander")
    eval_rule(49, "Ball Joint Wear", apply_rule(evidence_cf(p['clunk'], (1-p['clunk'])), R), "IF Creak + Camber Off")
    eval_rule(50, "Power Steering Pump Cavitation", apply_rule(evidence_cf(p['whine'], (1-p['whine'])), R), "IF Whine + Fluid Low")
    eval_rule(51, "Rack and Pinion Leak", apply_rule(evidence_cf(p['hard_p'], (1-p['hard_p'])), R), "IF Hard Steer + Loss")
    eval_rule(52, "Tie Rod End Play", apply_rule(evidence_cf(p['shake'], (1-p['shake'])), R), "IF Shake Whl + Wear")
    eval_rule(53, "Wheel Bearing Growl", apply_rule(evidence_cf(p['shake'], (1-p['shake'])), R), "IF Hum + Vibration")
    eval_rule(54, "Sway Bar Link Snapped", apply_rule(evidence_cf(p['clunk'], (1-p['clunk'])), R), "IF Roll + Knock")

    # --- FUEL SYSTEM (55-60) ---
    eval_rule(55, "Fuel Filter Clogged", apply_rule(evidence_cf(p['stall'], (1-p['stall'])), R), "IF Hesitation + Lean")
    eval_rule(56, "Fuel Pressure Regulator Failed", apply_rule(evidence_cf(p['smoke_black'], (1-p['smoke_black'])), R), "IF Black Smoke + Gas in Vac")
    eval_rule(57, "Fuel Tank Vent Blocked", apply_rule(evidence_cf(p['stall'], (1-p['stall'])), R), "IF Tank Hiss + Stall")
    eval_rule(58, "Flex Fuel Sensor Error", apply_rule(evidence_cf(p['slow_crank'], (1-p['slow_crank'])), R), "IF Hard Start + Alc Err")
    eval_rule(59, "Lift Pump Failure", apply_rule(evidence_cf(p['slow_crank'], (1-p['slow_crank'])), R), "IF Long Crank + Low Prime")
    eval_rule(60, "Fuel Line Restriction", apply_rule(evidence_cf(p['stall'], (1-p['stall'])), R), "IF Lean Code + Kink")

    # --- COOLING SYSTEM (61-65) ---
    eval_rule(61, "Radiator Strangled", apply_rule(evidence_cf(p['temp'], (1-p['temp'])), R), "IF Temp + Flow Blocked")
    eval_rule(62, "Water Pump Stopped", apply_rule(evidence_cf(p['temp'], (1-p['temp'])), R), "IF Temp + No Flow")
    eval_rule(63, "Internal Coolant Escape", apply_rule(evidence_cf(p['coolant_loss'], (1-p['coolant_loss'])), R), "IF Disappears + No Leak")
    eval_rule(64, "Heat Circulation Broken", apply_rule(evidence_cf(p['temp'], (1-p['temp'])), R), "IF No Heat + Coolant Low")
    eval_rule(65, "Heater Core Blocked", apply_rule(evidence_cf(p['temp'], (1-p['temp'])), R), "IF No Heat + Core Blocked")

    # --- COMBINED CONSPIRACIES (66-73) ---
    eval_rule(66, "Alternator Sabotage", apply_rule(evidence_cf(p['eng_light'], p['bat_light'], p['rpm_drop']), R), "IF Eng Light + Bat Light + RPM Unstable")
    eval_rule(67, "Shared Sensor Failure", apply_rule(evidence_cf(p['brk_light'], p['abs_light'], p['wheel_err']), R), "IF Brake Light + ABS Light + Wheel Err")
    eval_rule(68, "Head Gasket Breach", apply_rule(evidence_cf(p['temp'], p['smoke_white'], p['coolant_loss']), R), "IF Temp High + White Smoke + Coolant Loss")
    eval_rule(69, "System Paralysis", apply_rule(evidence_cf(p['abs_fault'], p['traction_fault'], p['wheel_err']), R), "IF ABS + Traction + Wheel Err")
    eval_rule(70, "Charging System Collapse", apply_rule(evidence_cf(p['dead_bat'], p['slow_crank'], p['volt']), R), "IF Dead Bat + Slow Crank + Volt Low")
    eval_rule(71, "Fuel/Ignition Conspiracy", apply_rule(evidence_cf(p['misfire'], (1-p['misfire']), p['trim']), R), "IF Misfire + O2 Flat + Trim High")
    eval_rule(72, "Engine/Trans Co-Suspects", apply_rule(evidence_cf(p['eng_light'], p['trans_temp'], p['rpm_drop']), R), "IF Eng Light + Trans Temp + RPM Drop")
    eval_rule(73, "Engine Internal Destruction", apply_rule(evidence_cf(p['oil_p'], p['knock'], p['temp']), R), "IF Oil Low + Knock + Temp High")

    trace.append("▶️ **INFERENCE COMPLETE.** Calculating final Certainty Factors...")

    # ==========================================
    # 5. RANKING, DISPLAY & EXPLANATION FACILITY
    # ==========================================
    st.divider()
    st.header("📋 Diagnostic Results")

    # Explanation Facility: HOW the conclusion was reached
    with st.expander("🧠 View System Reasoning Trace (How I thought)", expanded=False):
        for line in trace:
            st.text(line)

    final_report = []
    for fault, cf_list in results.items():
        score = combine_cf(cf_list)
        final_report.append({"fault": fault, "score": score, "logic": rule_text[fault], "multi": len(cf_list)})

    ranked = sorted(final_report, key=lambda x: x['score'], reverse=True)

    if not ranked:
        st.success("Analysis Complete: No strong evidence found for a specific failure based on current input.")
    else:
        top_score = ranked[0]['score']

        for item in ranked:
            score_pct = item['score'] * 100
            if item['score'] == top_score:
                st.error(f"### 🚨 PRIMARY DIAGNOSIS ({score_pct:.1f}%) ➔ {item['fault']}")
                st.code(f"Primary Logic Triggered: {item['logic']}", language="text")
                if item['multi'] > 1:
                    st.caption(f"ℹ️ Certainty boosted because {item['multi']} separate logical rules were triggered and combined.")
            elif item['score'] >= 0.4:
                st.warning(f"**Secondary Suspect ({score_pct:.1f}%)** ➔ {item['fault']}")
            else:
                pass # Hide low confidence ones to keep UI clean

Writing app.py


In [2]:
import os
import subprocess
import requests

# Download cloudflared.exe
url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-windows-amd64.exe"

response = requests.get(url)

with open("cloudflared_new.exe", "wb") as f:
    f.write(response.content)
    
print("Cloudflared downloaded successfully!")

# Start Streamlit app
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py"],
    shell=True
)

print("Streamlit started!")

# Start Cloudflare tunnel
cloudflare_process = subprocess.Popen(
    ["cloudflared.exe", "tunnel", "--url", "http://localhost:8501"],
    shell=True
)

Cloudflared downloaded successfully!
Streamlit started!
